In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("/Users/alexgonzalez/Documents/nba_quant")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
s20 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2019-20/regular_season/player_gamelogs.parquet')
s21 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2020-21/regular_season/player_gamelogs.parquet')
s22 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2021-22/regular_season/player_gamelogs.parquet')
s23 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2022-23/regular_season/player_gamelogs.parquet')
s24 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2023-24/regular_season/player_gamelogs.parquet')
s25 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2024-25/regular_season/player_gamelogs.parquet')
s26 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2025-26/regular_season/player_gamelogs.parquet')
s27 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2026-27/regular_season/player_gamelogs.parquet')
df = pd.concat([s20, s21, s22, s23, s24, s25, s26, s27]).reset_index(drop=True)
df['starting'] = (
    df['start_position'].astype('string').str.strip().isin(['G', 'F', 'C']).astype(int))
print(f"Number of rows: {len(df)}")
print(f"Number of columns: {len(df.columns)}")
print(df.columns.tolist())
print(df.dtypes)
print(df.head().to_string())

Number of rows: 178335
Number of columns: 190
['season_year', 'player_id', 'player_name', 'nickname', 'team_id', 'team_abbreviation', 'team_name', 'game_id', 'game_date', 'matchup', 'wl', 'min', 'fgm', 'fga', 'fg_pct', 'fg3_m', 'fg3_a', 'fg3_pct', 'ftm', 'fta', 'ft_pct', 'oreb', 'dreb', 'reb', 'ast', 'tov', 'stl', 'blk', 'blka', 'pf', 'pfd', 'pts', 'plus_minus', 'nba_fantasy_pts', 'dd2', 'td3', 'wnba_fantasy_pts', 'fp_high_score', 'available_flag', 'min_sec', 'team_count', 'e_off_rating', 'off_rating', 'sp_work_off_rating', 'e_def_rating', 'def_rating', 'sp_work_def_rating', 'e_net_rating', 'net_rating', 'sp_work_net_rating', 'ast_pct', 'ast_to', 'ast_ratio', 'oreb_pct', 'dreb_pct', 'reb_pct', 'tm_tov_pct', 'e_tov_pct', 'efg_pct', 'ts_pct', 'usg_pct', 'e_usg_pct', 'e_pace', 'pace', 'pace_per40', 'sp_work_pace', 'pie', 'poss', 'fgm_pg', 'fga_pg', 'team_city', 'team_tricode', 'team_slug', 'first_name', 'family_name', 'name_i', 'player_slug', 'start_position', 'comment', 'jersey_num', 'mi

### Bayes' Rule

In this example we will be using bayes' rule, we will use brandon miller with a line of 25.5 (points)

In [ ]:
PLAYER_NAME = 'Brandon Miller'
LINE = 25.5

data = df[df['player_name'] == PLAYER_NAME]
over = data["pts"] > LINE
start = data["starting"] == 1

# direct conditional
p_over_given_start = over[start].mean()

# same answer via Bayes' rule, as a check
p_start_given_over = start[over].mean()      # P(S | E)
p_over = over.mean()                         # P(E)
p_start = start.mean()                       # P(S)
p_bayes = p_start_given_over * p_over / p_start

print(p_over_given_start, p_bayes, start.sum())


0.21818181818181817 0.21818181818181817 165


In [19]:
from scipy import stats

k = over[start].sum()      # games over the line as a starter
n = start.sum()            # games as a starter

# prior: league-wide base rate for similar players, worth ~20 games of evidence
prior_rate, prior_strength = 0.15, 20
a0 = prior_rate * prior_strength
b0 = (1 - prior_rate) * prior_strength

posterior = stats.beta(a0 + k, b0 + n - k)
print(posterior.mean(), posterior.interval(0.9))

0.21081081081081082 (np.float64(0.16344354714335044), np.float64(0.26174102927822773))


In [20]:
import numpy as np
from scipy import stats, optimize

THRESH = 15.5
g = (
    df.assign(over=df["pts"] > THRESH)
      .groupby("player_name")
      .agg(k=("over", "sum"), n=("over", "size"))
)

# Learn (a, b) by maximizing the beta-binomial marginal likelihood across players
def nll(log_ab):
    a, b = np.exp(log_ab)
    return -stats.betabinom.logpmf(g["k"], g["n"], a, b).sum()

res = optimize.minimize(nll, x0=np.log([2.0, 6.0]), method="Nelder-Mead")
a0, b0 = np.exp(res.x)
print(f"population prior: Beta({a0:.2f}, {b0:.2f}), "
      f"mean {a0/(a0+b0):.3f}, strength {a0+b0:.1f} games")

# Each player's posterior: same conjugate update as before, but with the learned prior
g["raw_rate"] = g["k"] / g["n"]
g["post_mean"] = (a0 + g["k"]) / (a0 + b0 + g["n"])
g["post_lo"], g["post_hi"] = stats.beta(a0 + g["k"], b0 + g["n"] - g["k"]).interval(0.9)

population prior: Beta(0.47, 2.50), mean 0.159, strength 3.0 games


In [21]:
import pymc as pm
import pandas as pd

idx, players = pd.factorize(g.index)

with pm.Model() as model:
    mu = pm.Normal("mu", 0, 1.5)               # population mean (logit scale)
    sigma = pm.HalfNormal("sigma", 1.0)        # between-player spread
    z = pm.Normal("z", 0, 1, shape=len(players))   # non-centered player offsets
    theta = pm.Deterministic("theta", pm.math.sigmoid(mu + sigma * z))
    pm.Binomial("k", n=g["n"].to_numpy(), p=theta, observed=g["k"].to_numpy())
    idata = pm.sample(1000, tune=1000, target_accept=0.9)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu, sigma, z]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 17 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
